# Answer Types Reference

The questionnaire CSV/XLSX has 9 columns (A–I):

```
ID | Category | SubCategory | Question | AnswerType | AnswerOptions | Explanation | ParentID | TriggerValue
```

Options within a cell use `|` as a separator (pipe character).

---

## 1. `YES_NO`

Shorthand — auto-converted to `MULTIPLE_CHOICE_SINGLE` with `YES|NO`.

| Field | Value |
|---|---|
| AnswerType | `YES_NO` |
| AnswerOptions | *(leave blank — auto-filled)* |

**DB stored as:** `YES` or `NO`

---

## 2. `MULTIPLE_CHOICE_SINGLE`

Radio buttons — pick exactly one.

| Field | Value |
|---|---|
| AnswerType | `MULTIPLE_CHOICE_SINGLE` |
| AnswerOptions | `Good|Fair|Poor|Very Poor` |

**DB stored as:** the selected option text, e.g. `Fair`

---

## 3. `MULTIPLE_CHOICE_MULTI`

Checkboxes — pick one or more.

| Field | Value |
|---|---|
| AnswerType | `MULTIPLE_CHOICE_MULTI` |
| AnswerOptions | `Leakage|Corrosion|Blockage|Structural damage` |

**DB stored as:** pipe-separated selections, e.g. `Leakage|Blockage`

---

## 4. `INTEGER` / `DECIMAL` / `PERCENTAGE`

Shorthands for `NUMBER` — auto-converted.

| Type | AnswerOptions format | Example |
|---|---|---|
| `INTEGER` | `min\|max` or `min\|max\|unit1;unit2` | `0\|500` |
| `DECIMAL` | `min\|max` or `min\|max\|unit1;unit2` | `0\|100\|m;km` |
| `PERCENTAGE` | *(leave blank — bounds 0–100 auto-set)* | |

**DB stored as:** `42` or `3.5` or `7.2|km` (value|unit if units given)

---

## 5. `NUMBER`

Same as the shorthands above but the sub-format is specified explicitly in AnswerOptions.

| AnswerOptions format | Example |
|---|---|
| `INTEGER\|min\|max` | `INTEGER\|0\|200` |
| `DECIMAL\|min\|max` | `DECIMAL\|0.0\|9.9` |
| `DECIMAL\|min\|max\|unit1;unit2` | `DECIMAL\|0\|100\|m;km` |
| `PERCENTAGE` | `PERCENTAGE` |

**DB stored as:** `42` or `7.2|km`

---

## 6. `TEXT`

Free-form multi-line text. No options needed.

| Field | Value |
|---|---|
| AnswerType | `TEXT` |
| AnswerOptions | *(leave blank)* |

**DB stored as:** the typed text string

---

## 7. `DATE_YEAR`

Spinner of years.

| Field | Value |
|---|---|
| AnswerType | `DATE_YEAR` |
| AnswerOptions | `1950\|2030` (minYear\|maxYear) |

**DB stored as:** `2019`

---

## 8. `DATE_YEAR_MONTH`

Year + month spinners.

| Field | Value |
|---|---|
| AnswerType | `DATE_YEAR_MONTH` |
| AnswerOptions | `2000\|2030` |

**DB stored as:** `2023-06`

---

## 9. `DATE_FULLDATE`

Date picker dialog. No options.

| Field | Value |
|---|---|
| AnswerType | `DATE_FULLDATE` |
| AnswerOptions | *(leave blank)* |

**DB stored as:** `2024-03-15`

---

## 10. `TIME`

24-hour time picker. No options.

| Field | Value |
|---|---|
| AnswerType | `TIME` |
| AnswerOptions | *(leave blank)* |

**DB stored as:** `14:30`

---

## 11. `DATE_TIME`

Combined date + time picker.

| Field | Value |
|---|---|
| AnswerType | `DATE_TIME` |
| AnswerOptions | *(leave blank)* |

**DB stored as:** `2024-03-15 14:30`

---

## 12. `DURATION`

Numeric value + unit spinner.

Options format: `unit1|unit2;min;max` — semicolons separate the three sections; all parts optional.

| AnswerOptions | Behaviour |
|---|---|
| *(blank)* | All 7 default units (seconds/minutes/hours/days/weeks/months/years), no bounds |
| `hours\|days\|weeks` | Only these 3 units, no bounds |
| `minutes\|hours;1;240` | minutes or hours, bounded 1–240 |

**DB stored as:** `5|minutes`

---

## 13. `LOCATION`

GPS coordinates + address fields. AnswerOptions controls which fields appear.

Available field names: `COORDINATES`, `PROVINCE`, `DISTRICT`, `TEHSIL`, `VILLAGE`, `LOCALITY`

| AnswerOptions | Fields shown |
|---|---|
| *(blank)* | All: coordinates + province/district/tehsil/village/locality |
| `COORDINATES` | Lat/lng + map picker only |
| `COORDINATES\|DISTRICT\|TEHSIL` | Coordinates + district + tehsil |

**DB stored as:** 7 pipe-separated slots in fixed order:

```
lat|lng|village|tehsil|district|province|locality
```

Example: `33.9891|71.5234|Hayatabad|Peshawar|Peshawar|KPK|`

---

## 14. `LINE` / `POLYGON`

Geometry drawn on an interactive map. No options.

| Field | Value |
|---|---|
| AnswerType | `LINE` or `POLYGON` |
| AnswerOptions | *(leave blank)* |

**DB stored as:** JSON array of coordinate pairs, e.g.:

```json
[[33.98,71.52],[33.99,71.53],[33.97,71.54]]
```

---

## 15. `COMPOUND`

Multiple typed sub-fields grouped inside one question card.

Options format (pipe-separated field definitions, colon-separated tokens within each):

```
label:type[:subtype[:min[:max[:unit1;unit2]]]]
```

| AnswerOptions example | Result |
|---|---|
| `Pipe Diameter:NUMBER:INTEGER:0:1000\|Material:TEXT\|Age:NUMBER:INTEGER:0:100` | Three sub-fields |

Sub-field types supported inside COMPOUND: `TEXT`, `NUMBER` (with optional `INTEGER`/`DECIMAL`/`PERCENTAGE` subtype, bounds, and units)

**DB stored as:** `label=value||label2=value2||...`

Example: `Pipe Diameter=150||Material=Steel||Age=12`

---

## Sub-questions (conditional follow-ups)

Any row with a `ParentID` and `TriggerValue` becomes a sub-question, shown only when the parent answer matches the trigger.

```
ID  | Cat | SubCat | Question           | AnswerType             | AnswerOptions | Explanation | ParentID | TriggerValue
101 |  …  |   …    | Pipe condition?    | MULTIPLE_CHOICE_SINGLE | Good|Bad      |             |          |
102 |  …  |   …    | Describe the issue | TEXT                   |               |             | 101      | Bad
```

Sub-questions can use any answer type. Multiple sub-questions can share the same `TriggerValue`.
